# Task 2.2 - CNN Image Classifier
**Skill Set Go EduTech - AI/ML Internship - Week 2**

**Goal:** prepare an image dataset and train a convolutional neural network in PyTorch to classify images.

**Dataset:** CIFAR-10 - 60,000 colour images (32x32 pixels, 3 channels) in 10 classes.
It is small enough to train on Colab and has clear class labels. (Set `USE_SUBSET = True` below for a quick test run.)

In [ ]:
import os, json, time
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as T
from sklearn.metrics import confusion_matrix, classification_report

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
os.makedirs("results", exist_ok=True)

USE_SUBSET = False     # True = only 10,000 training images (fast test run)
EPOCHS = 15
BATCH_SIZE = 64
NUM_WORKERS = 2

## 1. Inspect the data first
Before training: image size, channels, and class balance.

In [ ]:
raw_train = torchvision.datasets.CIFAR10("./data", train=True, download=True)   # PIL images, no transform
classes = raw_train.classes
print("Classes:", classes)
print("Number of training images:", len(raw_train))

img, label = raw_train[0]
print("One image -> size:", img.size, "| mode (channels):", img.mode, "| label:", classes[label])

counts = np.bincount(raw_train.targets)
for c, n in zip(classes, counts):
    print(f"{c:12s} {n}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for i, ax in enumerate(axes.ravel()):
    im, lb = raw_train[i]
    ax.imshow(im)
    ax.set_title(classes[lb], fontsize=9)
    ax.axis("off")
plt.suptitle("Sample CIFAR-10 training images")
plt.tight_layout()
plt.savefig("results/sample_images.png", dpi=150)
plt.show()

**Observation:** every class has exactly 5,000 training images, so the dataset is perfectly balanced and class
imbalance is not a problem here. Images are tiny (32x32) RGB, so they look blurry and some classes
(cat/dog, deer/horse) can be hard to tell apart.

## 2. Preprocessing, augmentation and splits
* **Training images:** random horizontal flip + random crop (with padding) as light augmentation, then `ToTensor` and
  normalisation with the CIFAR-10 channel means/std.
* **Validation and test images:** only `ToTensor` + the same normalisation (no random augmentation).
* 45,000 images for training, 5,000 for validation (chosen randomly with a fixed seed); the official 10,000 test images
  are kept aside for the final check.

In [ ]:
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2470, 0.2435, 0.2616)
IMAGE_SIZE = 32

train_tf = T.Compose([T.RandomHorizontalFlip(), T.RandomCrop(32, padding=4), T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf  = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])

train_aug  = torchvision.datasets.CIFAR10("./data", train=True,  download=False, transform=train_tf)
train_eval = torchvision.datasets.CIFAR10("./data", train=True,  download=False, transform=eval_tf)
test_set   = torchvision.datasets.CIFAR10("./data", train=False, download=True,  transform=eval_tf)

g = torch.Generator().manual_seed(42)
perm = torch.randperm(50000, generator=g).tolist()
train_idx, val_idx = perm[:45000], perm[45000:]
if USE_SUBSET:
    train_idx, val_idx = train_idx[:10000], val_idx[:2000]

train_set = Subset(train_aug, train_idx)      # augmented
val_set   = Subset(train_eval, val_idx)       # same images source, but no augmentation

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=NUM_WORKERS)

print("train / val / test sizes:", len(train_set), len(val_set), len(test_set))
xb, yb = next(iter(train_loader))
print("One batch:", xb.shape, yb.shape)     # (64, 3, 32, 32), (64,)

## 3. The CNN
Three convolution blocks (Conv -> ReLU -> MaxPool), then a small fully-connected classifier.
I kept it small on purpose - CIFAR-10 does not need a huge model, and a bigger one would train slower and overfit more.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 32x32 -> 16x16
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16x16 -> 8x8
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 8x8 -> 4x4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                     # 128 * 4 * 4 = 2048 values
            nn.Dropout(0.3),
            nn.Linear(128 * 4 * 4, 128), nn.ReLU(),
            nn.Linear(128, num_classes),      # raw scores for the 10 classes
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = SmallCNN(num_classes=len(classes)).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print("Parameters:", n_params)

### How the image dimensions change through the network

In [ ]:
x = torch.zeros(1, 3, 32, 32).to(device)
print("input          :", tuple(x.shape))
for layer in model.features:
    x = layer(x)
    if isinstance(layer, (nn.Conv2d, nn.MaxPool2d)):
        print(f"{layer.__class__.__name__:14s} :", tuple(x.shape))
x = model.classifier[0](x)
print("Flatten        :", tuple(x.shape))

## 4. Training
Cross-entropy loss (multi-class), Adam optimizer (lr = 0.001). Training and validation loss/accuracy are recorded every
epoch, and the weights with the best **validation** accuracy are kept.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def evaluate(loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    preds_all, labels_all = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            total_loss += criterion(out, yb).item() * xb.size(0)
            p = out.argmax(dim=1)
            correct += (p == yb).sum().item()
            n += xb.size(0)
            preds_all.append(p.cpu()); labels_all.append(yb.cpu())
    return total_loss / n, correct / n, torch.cat(preds_all).numpy(), torch.cat(labels_all).numpy()

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc, best_state = 0.0, None
start = time.time()

for epoch in range(EPOCHS):
    model.train()
    run_loss, run_correct, seen = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        run_loss += loss.item() * xb.size(0)
        run_correct += (out.argmax(dim=1) == yb).sum().item()
        seen += xb.size(0)

    val_loss, val_acc, _, _ = evaluate(val_loader)
    history["train_loss"].append(run_loss / seen)
    history["train_acc"].append(run_correct / seen)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | train loss {history['train_loss'][-1]:.3f} acc {history['train_acc'][-1]:.3f}"
          f" | val loss {val_loss:.3f} acc {val_acc:.3f}")

print(f"Training time: {time.time() - start:.0f}s | best validation accuracy: {best_val_acc:.4f}")
model.load_state_dict(best_state)   # go back to the best epoch

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history["train_loss"], label="Train")
ax[0].plot(history["val_loss"], label="Validation")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Cross-entropy loss"); ax[0].set_title("Loss per epoch"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(history["train_acc"], label="Train")
ax[1].plot(history["val_acc"], label="Validation")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy"); ax[1].set_title("Accuracy per epoch"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/cnn_curves.png", dpi=150)
plt.show()

## 5. Final evaluation on the test set
Test images were never used for training or for choosing the best epoch, so this is the honest number to report
(not the training accuracy).

In [ ]:
test_loss, test_acc, test_preds, test_labels = evaluate(test_loader)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")
print()
print(classification_report(test_labels, test_preds, target_names=classes, digits=3))

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(classes, rotation=45, ha="right")
ax.set_yticks(range(10)); ax.set_yticklabels(classes)
ax.set_xlabel("Predicted class"); ax.set_ylabel("True class"); ax.set_title("Confusion matrix (test set)")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im)
plt.tight_layout()
plt.savefig("results/confusion_matrix.png", dpi=150)
plt.show()

per_class = cm.diagonal() / cm.sum(axis=1)
for c, a in zip(classes, per_class):
    print(f"{c:12s} {a:.3f}")

### Correct and incorrect predictions

In [ ]:
raw_test = torchvision.datasets.CIFAR10("./data", train=False, download=False)   # PIL images for display

correct_idx = np.where(test_preds == test_labels)[0][:5]
wrong_idx   = np.where(test_preds != test_labels)[0][:5]

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for row, idxs, name in [(0, correct_idx, "Correct"), (1, wrong_idx, "Wrong")]:
    for col, i in enumerate(idxs):
        axes[row, col].imshow(raw_test[i][0])
        axes[row, col].set_title(f"{name}\ntrue: {classes[test_labels[i]]}\npred: {classes[test_preds[i]]}", fontsize=8)
        axes[row, col].axis("off")
plt.tight_layout()
plt.savefig("results/sample_predictions.png", dpi=150)
plt.show()

## 6. Save the model and the preprocessing needed to use it again
A saved model is useless without the exact preprocessing it was trained with (image size, mean, std, class order).

In [ ]:
torch.save(model.state_dict(), "results/cnn_cifar10.pth")

preprocessing = {"image_size": IMAGE_SIZE, "mean": list(MEAN), "std": list(STD), "classes": classes}
with open("results/preprocessing.json", "w") as f:
    json.dump(preprocessing, f, indent=2)

print("Model size (KB):", round(os.path.getsize("results/cnn_cifar10.pth") / 1024))

## 7. Load the saved model and classify new images
This starts from the saved files only (a fresh model object), the way someone else would use it.

In [ ]:
from PIL import Image

def load_for_inference(model_path="results/cnn_cifar10.pth", prep_path="results/preprocessing.json"):
    with open(prep_path) as f:
        prep = json.load(f)
    m = SmallCNN(num_classes=len(prep["classes"]))
    m.load_state_dict(torch.load(model_path, map_location="cpu"))
    m.eval()
    tf = T.Compose([T.Resize((prep["image_size"], prep["image_size"])), T.ToTensor(),
                    T.Normalize(prep["mean"], prep["std"])])
    return m, tf, prep["classes"]

def classify(pil_image, m, tf, class_names):
    x = tf(pil_image.convert("RGB")).unsqueeze(0)
    with torch.no_grad():
        probs = torch.softmax(m(x), dim=1)[0]
    top = int(probs.argmax())
    return class_names[top], float(probs[top])

loaded_model, loaded_tf, loaded_classes = load_for_inference()

for i in range(5):
    pil_img, true_label = raw_test[i]
    name, conf = classify(pil_img, loaded_model, loaded_tf, loaded_classes)
    print(f"test image {i}: true = {classes[true_label]:10s} | predicted = {name:10s} ({conf:.2f})")

# Optional: try your own photo (upload it to Colab first), e.g.
# my_img = Image.open("my_photo.jpg")
# print(classify(my_img, loaded_model, loaded_tf, loaded_classes))

In [ ]:
final_train_acc = history["train_acc"][int(np.argmax(history["val_acc"]))]
results = {
    "task": "2.2",
    "dataset": "CIFAR-10 (32x32 RGB, 10 classes)",
    "train_images": len(train_set), "val_images": len(val_set), "test_images": len(test_set),
    "model": "SmallCNN: 3 conv blocks (32-64-128) + FC(128) + dropout 0.3",
    "parameters": n_params,
    "optimizer": "Adam (lr=0.001)", "batch_size": BATCH_SIZE, "epochs": EPOCHS,
    "best_val_acc": round(best_val_acc, 4),
    "train_acc_at_best_epoch": round(final_train_acc, 4),
    "test_acc": round(float(test_acc), 4),
    "per_class_test_acc": {c: round(float(a), 3) for c, a in zip(classes, per_class)},
}
with open("results/cnn_results.json", "w") as f:
    json.dump(results, f, indent=2)
results

## 8. My notes on the results
*(Write in your own words after you have your output.)*

* Test accuracy vs. training accuracy - is there a gap? What does it suggest?
* Which classes are easiest / hardest? Which classes get confused with each other in the confusion matrix?
* What did the wrong predictions look like (blurry, unusual angle, similar class)?
* What would I try next (more augmentation, batch norm, more epochs)?

**My notes:**

-